# PEM hydrogen consumption rate

Goal: estimate how much hydrogen is consumed when the PEM delivers electrical energy.

Important: the volume reading was taken at the end of the full discharge test, not exactly at the EMS
cutoff voltage. Therefore I keep two ideas separate:

- EMS cutoff voltage: used to decide when the fuel cell should stop operating normally.
- end volume reading: used to estimate total hydrogen consumed in the full discharge experiment.

For varying EMS loads, mL/J is the most useful rate because load demand is a power signal.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes the notebook work both from the repo root and from its own folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find the project root folder containing data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Load and correct the full PEM discharge test


In [ ]:
full_file = PROJECT_ROOT / "data/PEM_test/charge_discharge/discharge_PEM_full.csv"
full_raw = pd.read_csv(full_file)

full = full_raw.copy()
full["timestamp"] = pd.to_datetime(full["timestamp"])
full["time_s"] = (full["timestamp"] - full["timestamp"].iloc[0]).dt.total_seconds()
full["pem_voltage_V"] = full["ina4_bus_V"] - 0.064
full["pem_current_A"] = 0.843 * (full["ina4_current_mA"] / 1000) + 0.001
full["pem_output_current_A"] = (-full["pem_current_A"]).clip(lower=0)
full["pem_output_power_W"] = full["pem_voltage_V"].clip(lower=0) * full["pem_output_current_A"]

display(full.head())


## 2. Select the actual discharge period


In [ ]:
discharge = full[(full["scenario"] == 6) & (full["pem_output_current_A"] > 0.005)].copy()
discharge["discharge_time_s"] = discharge["time_s"] - discharge["time_s"].iloc[0]

print("Selected discharge rows:", len(discharge))
display(discharge[["discharge_time_s", "pem_voltage_V", "pem_output_current_A", "pem_output_power_W"]].head())


## 3. Plot the full discharge


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(discharge["discharge_time_s"], discharge["pem_voltage_V"], label="PEM voltage")
plt.axhline(0.54975, color="black", linestyle="--", label="EMS cutoff")
plt.axhline(0.20, color="tab:red", linestyle=":", label="approx. end-volume endpoint")
plt.title("Full PEM discharge voltage")
plt.xlabel("Discharge time [s]")
plt.ylabel("PEM voltage [V]")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(discharge["discharge_time_s"], discharge["pem_output_power_W"])
plt.title("Full PEM output power")
plt.xlabel("Discharge time [s]")
plt.ylabel("PEM output power [W]")
plt.grid(True)
plt.show()


## 4. Integrate output energy up to the volume-reading endpoint

The manual volume value was read at the end of the experiment. I therefore use the lower endpoint
around 0.20 V for the empirical consumption rate. This is not the same as the normal EMS cutoff.


In [ ]:
PEM_FULL_HYDROGEN_CAPACITY_ML = 15.6
H2_LEFT_AT_FULL_TEST_END_ML = 7.6
H2_USED_DURING_FULL_TEST_ML = PEM_FULL_HYDROGEN_CAPACITY_ML - H2_LEFT_AT_FULL_TEST_END_ML

ENDPOINT_VOLTAGE_FOR_VOLUME_READING_V = 0.20

endpoint_candidates = discharge[
    (discharge["discharge_time_s"] > 30)
    & (discharge["pem_voltage_V"] <= ENDPOINT_VOLTAGE_FOR_VOLUME_READING_V)
]

if endpoint_candidates.empty:
    selected_discharge = discharge.copy()
else:
    selected_discharge = discharge.loc[:endpoint_candidates.index[0]].copy()

selected_discharge["dt_s"] = selected_discharge["discharge_time_s"].diff().fillna(0)
OUTPUT_CHARGE_C = (selected_discharge["pem_output_current_A"] * selected_discharge["dt_s"]).sum()
OUTPUT_ENERGY_J = (selected_discharge["pem_output_power_W"] * selected_discharge["dt_s"]).sum()

HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_C = H2_USED_DURING_FULL_TEST_ML / OUTPUT_CHARGE_C
HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_J = H2_USED_DURING_FULL_TEST_ML / OUTPUT_ENERGY_J

print(f"Hydrogen used in full test: {H2_USED_DURING_FULL_TEST_ML:.2f} mL")
print(f"Integrated output charge:   {OUTPUT_CHARGE_C:.2f} C")
print(f"Integrated output energy:   {OUTPUT_ENERGY_J:.2f} J")
print(f"Consumption rate:           {HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_J:.4f} mL/J")


## 5. Compare with a cutoff-only integration


In [ ]:
PEM_MIN_USABLE_VOLTAGE = 0.54975

cutoff_candidates = discharge[
    (discharge["discharge_time_s"] > 30)
    & (discharge["pem_voltage_V"] <= PEM_MIN_USABLE_VOLTAGE)
]
cutoff_discharge = discharge.loc[:cutoff_candidates.index[0]].copy()
cutoff_discharge["dt_s"] = cutoff_discharge["discharge_time_s"].diff().fillna(0)
cutoff_energy_J = (cutoff_discharge["pem_output_power_W"] * cutoff_discharge["dt_s"]).sum()

print(f"Output energy up to EMS cutoff only: {cutoff_energy_J:.2f} J")
print(
    "Do not use the end-volume reading with this cutoff unless you also know how much hydrogen was left at the cutoff."
)


## 6. Values to use in the app


In [ ]:
discharge_parameters = pd.DataFrame(
    {
        "parameter": [
            "PEM_FULL_HYDROGEN_CAPACITY_ML",
            "H2_LEFT_AT_FULL_TEST_END_ML",
            "H2_USED_DURING_FULL_TEST_ML",
            "HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_J",
            "HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_C",
            "ENDPOINT_VOLTAGE_FOR_VOLUME_READING_V",
        ],
        "value": [
            PEM_FULL_HYDROGEN_CAPACITY_ML,
            H2_LEFT_AT_FULL_TEST_END_ML,
            H2_USED_DURING_FULL_TEST_ML,
            HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_J,
            HYDROGEN_CONSUMPTION_ML_PER_OUTPUT_C,
            ENDPOINT_VOLTAGE_FOR_VOLUME_READING_V,
        ],
        "unit": ["mL", "mL", "mL", "mL/J", "mL/C", "V"],
        "meaning": [
            "Measured full tank amount",
            "Manual reading at the end of full test",
            "Hydrogen consumed in the full experiment",
            "Best rate for variable power EMS simulation",
            "Useful for current-based checks",
            "Endpoint matched to the manual volume reading",
        ],
    }
)

display(discharge_parameters)
